In [3]:
import torch 


# input of layer 2
bs = 16 # batch size

Leye = torch.rand(bs,128, 8, 8)
print(Leye)
Reye = torch.rand(bs,128, 8, 8)
print(Reye)
FaceData = torch.rand(bs,128, 8, 8)  # currently matched with eyes......
print(FaceData)

tensor([[[[6.4812e-01, 9.6390e-01, 1.1203e-02,  ..., 9.5749e-01,
           6.4386e-02, 3.8508e-01],
          [6.6852e-01, 8.5083e-01, 3.8782e-01,  ..., 4.5384e-01,
           9.6458e-01, 7.6756e-01],
          [6.6795e-01, 1.5003e-01, 2.6110e-01,  ..., 3.5176e-01,
           4.8976e-01, 8.7292e-02],
          ...,
          [8.5445e-01, 3.3251e-01, 5.4146e-01,  ..., 8.8547e-01,
           9.7570e-01, 6.4901e-01],
          [9.8144e-01, 8.8869e-01, 4.5328e-02,  ..., 2.9592e-01,
           6.7479e-01, 3.9114e-02],
          [1.7735e-01, 3.6701e-01, 4.3448e-01,  ..., 5.5356e-01,
           5.5539e-01, 4.7000e-01]],

         [[9.8260e-02, 8.3244e-01, 1.5103e-01,  ..., 3.7159e-02,
           3.9529e-02, 6.5978e-01],
          [3.0783e-01, 2.4109e-01, 6.0092e-01,  ..., 8.6829e-01,
           6.1251e-01, 4.5027e-02],
          [7.7436e-01, 9.3036e-01, 2.3669e-01,  ..., 9.3959e-01,
           7.9585e-01, 2.4555e-01],
          ...,
          [5.6372e-03, 1.7776e-01, 6.2651e-01,  ..., 1.4971

In [4]:
from vit_pytorch import ViT


class TinyModel(torch.nn.Module):
    def __init__(self, gaze_dims=2):    # the gaze_dims should be defined later, according to the dataset and what we wanna predict
        super(TinyModel, self).__init__()
        # layer 1: feature extraction (to be implemented)
        
        
        # layer 2: feature fusion: concate + group  normalization
        # self.concate = torch.cat((x, x, x), 0) // not needed here, only in forward
        # total_ch = Leye[1]+Reye[1]+FaceData[1]  # total channels of the input // this is not working 
        total_ch = 384
        self.gn = torch.nn.GroupNorm(3, total_ch)     # Separate 6 channels into 3 groups, how to define a appropriate number of groups & channels?
        
        # layer 3:self-attention
        self.vit = ViT(
                image_size = 8,
                patch_size = 2,
                num_classes = 3,  # for instance 3 gaze dims, PoG (x,y,z)
                dim = 1024,
                depth = 6,
                heads = 16,
                mlp_dim = 2048,
                channels= total_ch,
                dropout = 0.1,
                emb_dropout = 0.1)
        
        # can replace the mlp_head with a new one for regression tasks, or just simply change the num_classes to the number of gaze dims
        # self.vit.mlp_head = torch.nn.Sequential(
        #     torch.nn.Linear(1024, 512),
        #     torch.nn.ReLU(),
        #     torch.nn.Dropout(0.1),
        #     torch.nn.Linear(512, 256),
        #     torch.nn.ReLU(),
        #     torch.nn.Dropout(0.1),
        #     torch.nn.Linear(256, gaze_dims)
        # )
 
        # RNN layer for temporal information
        # https://pytorch.org/docs/stable/generated/torch.nn.GRU.html
        # torch.nn.GRU(input_size, hidden_size, num_layers=1, bias=True, batch_first=False, dropout=0.0, bidirectional=False, device=None, dtype=None)
        # how to define the input_size and hidden_size?
        self.rnn = torch.nn.GRU(1024, 512, num_layers=1, bias=True, batch_first=False, dropout=0.0, bidirectional=False, device=None, dtype=None)

    def forward(self, left_eye, right_eye, face):

        # layer2
        concate = torch.cat((left_eye, right_eye, face), 1)  # dim = 0 or 1?  only channel dim changes?
        out = self.gn(concate)
        bs, c, h, w = out.shape
        out = out.reshape(bs, c, h, w)
        out = self.vit(out)
        out = self.rnn(out)
        return out 
        

model = TinyModel(gaze_dims=3)
print(model)

output = model(Leye, Reye, FaceData)  # currently the face Data is not matched with eyes, should be matched in the future, how to match? linear padding or other methods? would it affect the performance if change the size of facedata
print("output:",output.shape)
# testing only forward pass, not backward pass


TinyModel(
  (gn): GroupNorm(3, 384, eps=1e-05, affine=True)
  (vit): ViT(
    (to_patch_embedding): Sequential(
      (0): Rearrange('b c (h p1) (w p2) -> b (h w) (p1 p2 c)', p1=2, p2=2)
      (1): LayerNorm((1536,), eps=1e-05, elementwise_affine=True)
      (2): Linear(in_features=1536, out_features=1024, bias=True)
      (3): LayerNorm((1024,), eps=1e-05, elementwise_affine=True)
    )
    (dropout): Dropout(p=0.1, inplace=False)
    (transformer): Transformer(
      (norm): LayerNorm((1024,), eps=1e-05, elementwise_affine=True)
      (layers): ModuleList(
        (0-5): 6 x ModuleList(
          (0): Attention(
            (norm): LayerNorm((1024,), eps=1e-05, elementwise_affine=True)
            (attend): Softmax(dim=-1)
            (dropout): Dropout(p=0.1, inplace=False)
            (to_qkv): Linear(in_features=1024, out_features=3072, bias=False)
            (to_out): Sequential(
              (0): Linear(in_features=1024, out_features=1024, bias=True)
              (1): Dropou

RuntimeError: input.size(-1) must be equal to input_size. Expected 1024, got 3